# AIM:
create evaluation workflow. Taking the manually extracted Ci impacts (validation set) and compare it with the CI impacts (llm_geolocations.ipynb) extrracted by the first LLM 1. 
As a first step the evaluation should be done only for the direct CI impacts - CI type, damage and geolocation

Issue:
* What is needed an approach that recognizes when an direct impact case is not detected by the model
Idea: 
* Split the original texts passed to the model on the exact chunks as again
* Then chunkwise check if the CI impacts from the validation set correspond in number and their textual similarity to the CI impacts infered by the LLM 1 and Entity Linking 

## Semantic Textual Similarity (STS)

Calculating the STS for both model configurations (chain of prompts, orchestration of models)
The outputs are cosine similarity scores for similar model outputs per chunk. They are ranked by score for each model, restricted to the top 20 results.  


In [1]:
import os
import sys
import io
import numpy as np

from pathlib import Path
import pickle
import time
import warnings
import subprocess
import importlib

import spacy
import pandas as pd
import torch
import pyarrow as pa
import pyarrow.parquet as pq

sys.path.append("../")
from src.settings import settings as s
import src.document_cleaning as dc

a-buch-ThinkPad-X1-Extreme-Gen-4i
Hostname: CompletedProcess(args=['hostname'], returncode=0) None
Running on TUB Cluster


/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Direct CI impacts: LLM 1 vs domain-expertise 

In [20]:
#  Suppress future warnings from PyTorch
warnings.filterwarnings("ignore", category=FutureWarning)

#  Define data dir where tags.csv and domain-expertise derived tag lists are found 
PATH_MANUAL_DATA = Path("../" + s.PATH_EVALUATION + 'manual_extracted')
PATH_EVAL_RESULT = Path("../" + s.PATH_DATA + 'evaluation_results')
os.makedirs(PATH_EVAL_RESULT, exist_ok=True)
LX_DATA_FILEPATH = "../" / Path(s.PATH_LX_DATA / s.LX_DATA_FILENAME)

df_valid = pd.read_csv(
    PATH_MANUAL_DATA / 'table_ci_impacts_sm.csv',
    usecols=["publication_id", "ci1_type", "ci1_damage", "ci1_location"]
)

print(len(df_valid))
## pre-process: 
# remove undone entries
df_valid = df_valid[~df_valid["publication_id"].str.contains("xx")]
print(len(df_valid))


## load predictions

# NOTE workaround due that LX was not saved to CSV

## fix missing saving of LX to CSV
# Solving by conversion of jsonl to csv

df_pred = pd.read_csv(
    LX_DATA_FILEPATH,
    usecols=["citation_id", "infrastructure_type", "damage", "geolocation"]
)
df_pred

131
122


,infrastructure_type,damage,geolocation,citation_id
0,bridges,destroyed,Ahr valley,Koks et al 2022 - Brief communication_cleaned
1,motorways,closure,NAN,Koks et al 2022 - Brief communication_cleaned
2,motorways,closure,NAN,Koks et al 2022 - Brief communication_cleaned
3,bridges,destroyed,Ahr valley,Koks et al 2022 - Brief communication_cleaned
4,sewage systems,completely destroyed,NAN,Koks et al 2022 - Brief communication_cleaned
...,...,...,...,...
6767,NAN,NAN,NAN,Yamashita 2008 - Analysis control and econom...
6768,NAN,NAN,NAN,Yamashita 2008 - Analysis control and econom...
6769,NAN,NAN,NAN,Yamashita 2008 - Analysis control and econom...
6770,motorways,closure,NAN,Yamashita 2008 - Analysis control and econom...


#### Create vectors for evaluation and LLm output

In [126]:
## load english model with word vectors included
s.SPACY_MODEL = "en_core_web_lg"

try: 
    print("Try loading spaCy language model for local machine ...")
    try: 
        nlp = spacy.load(s.SPACY_MODEL)
    except (OSError, ValueError):
        print(f"spaCy language model '{s.SPACY_MODEL}' not found. Downloading ...")
        ## loading transformer language model for NER requires additional package
        if (
            s.SPACY_MODEL.endswith("_trf")
            and importlib.util.find_spec("spacy[transformers]") is None
        ):
            !uv add spacy[transformers]
        !uv run python -m spacy download {s.SPACY_MODEL}
        nlp = spacy.load(s.SPACY_MODEL)

except Exception as e:
    print("Try loading spaCy language model for remote instance (e.g., cluster)")
    try: 
        nlp = spacy.load(s.SPACY_MODEL)
    except (OSError, ValueError):
        print(f"spaCy language model '{s.SPACY_MODEL}' not found. Downloading ...")
        subprocess.check_call(["uv", "pip", "install", "spacy-transformers"])
        subprocess.check_call(["uv", "run", "python", "-m", "spacy", "download", s.SPACY_MODEL])
        nlp = spacy.load(s.SPACY_MODEL)
        


Try loading spaCy language model for local machine ...


In [127]:
def cosine_similarity(vector_a, vector_b):
    dot_product = np.dot(vector_a, vector_b)
    magnitude_a = np.linalg.norm(vector_a)
    magnitude_b = np.linalg.norm(vector_b)
    return dot_product / (magnitude_a * magnitude_b)

# Access the vector for specific word(s)
vec_estimated = nlp(df_valid.ci1_damage[1]).vector
vec_valid = nlp(df_pred.damage[1]).vector
ci_impact_similarity = cosine_similarity(vec_estimated, vec_valid)  # 0-1 value, the higher the more similar
ci_impact_similarity



np.float32(0.6684598)

In [22]:
df_pred

,infrastructure_type,damage,geolocation,citation_id
0,bridges,destroyed,Ahr valley,Koks et al 2022 - Brief communication_cleaned
1,motorways,closure,NAN,Koks et al 2022 - Brief communication_cleaned
2,motorways,closure,NAN,Koks et al 2022 - Brief communication_cleaned
3,bridges,destroyed,Ahr valley,Koks et al 2022 - Brief communication_cleaned
4,sewage systems,completely destroyed,NAN,Koks et al 2022 - Brief communication_cleaned
...,...,...,...,...
6767,NAN,NAN,NAN,Yamashita 2008 - Analysis control and econom...
6768,NAN,NAN,NAN,Yamashita 2008 - Analysis control and econom...
6769,NAN,NAN,NAN,Yamashita 2008 - Analysis control and econom...
6770,motorways,closure,NAN,Yamashita 2008 - Analysis control and econom...


In [45]:
from contextlib import suppress
import re


filename = "Koks et al 2022 - blubb nd cleaned"

citation_pattern = r"(.*?)(\d{4})(.*)" # split at first occurrence of year
with suppress(AttributeError):
    authors, year = re.findall(citation_pattern, filename)[0][:2] 
# with suppress(AttributeError):
authors = authors.replace("et al ", "")
authors
citation = f"{authors} {year}"
citation = citation.replace("  ", " ").strip()
citation


'Koks 2022'

In [ ]:


## get same impact entries
columns_valid = ["ci1_type", "ci1_damage", "ci1_location"]
columns_pred = ["infrastructure_type", "damage", "location"]

for column_valid, column_pred in zip(columns_valid, columns_pred):

    print(f" --------- Processing column pair: {column_valid} - {column_pred} ------------")
    
    df_valid_pred_all = pd.DataFrame()
    citations_list = []

    ## for each entry in df_pred
    for i in range(len(df_pred)):
        
        highest_similarity_score = 0.00
        
        ## needed to traceback info when entry is missing in valid. DS (eeg buildin impact fo EFE 2024)
        chunk_id_value_pred = df_pred.chunk_id[i]

        # get first entry of LLM prediction
        df_pred_doc = df_pred.iloc[i]
        citation_str = df_pred_doc.citation
        citations_list.append(citation_str)
        print("Searching for citation:", citation_str)


        # get all corresponding validation documents
        df_valid_entries = df_valid[df_valid["publication_id"].isin([citation_str])]

        ### # preprocessing:
        #  handle on NANs
        df_valid_entries[column_valid] = df_valid_entries[column_valid].astype(str)

        # remove double whitespaces
        # df_pred_doc[column_pred] = df_pred_doc[column_pred].replace("  ", " ")
        # df_valid_entries[column_valid] = df_valid_entries[column_valid].replace("  ", " ")


        # compute similarity between each predicted impact case and all validation impact cases (cross-product)
        impact_pred = df_pred_doc[column_pred]
        vec_pred = nlp(impact_pred).vector


        # print(f"Searching for highest similarity to `{impact_pred}` in validation set ... ")
        for j in range(len(df_valid_entries[column_valid])):

            if df_valid_entries[column_valid].iloc[j] == "nan":
                continue

            impact_valid = df_valid_entries[column_valid].iloc[j]

            vec_valid = nlp(impact_valid).vector
            similarity_score = cosine_similarity(vec_valid, vec_pred)  # 0-1 value, the higher the more similar
            # print(f"Similarity {i}-{j}: {similarity_score}")

            ## get only  pair with highest similarity
            if similarity_score > highest_similarity_score:
                # print("New score, old score", similarity_score, highest_similarity_score)
                highest_similarity_score = similarity_score
                dict_pair = {
                    "impact_pred": impact_pred, 
                    "impact_valid": impact_valid, 
                    "similarity": highest_similarity_score,
                    "citation": citation_str,
                    "chunk_id_pred": df_pred.chunk_id[i]
                }
            else:
                continue

        df_valid_pred_all = pd.concat([df_valid_pred_all, pd.DataFrame([dict_pair])], ignore_index=True)

    print(" ---------- Evaluation summary statistics: -----------")
    print(df_valid_pred_all.similarity.describe())



    SIMILARITY_FILENAME = f'lx1_similarity_{column_pred}.parquet'
    SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

    print("Saving evaluation statistics and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
    with open(SIMILARITY_FILEPATH, 'w') as f:
        # results
        df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
        pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)   
        #   summary statistics
        df_valid_pred_all_stats = df_valid_pred_all.describe()
        f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
        df_valid_pred_all_stats.to_json(f, indent=4)
        


    similarity_threshold = 0.75
    df_valid_pred_all['is_similar'] = df_valid_pred_all['similarity'] >= similarity_threshold
    print(f"Number of similar impact cases (similarity >= {similarity_threshold}): {df_valid_pred_all['is_similar'].sum()} out of {len(df_valid_pred_all)}")

    SIMILARITY_FILENAME = f'lx1_similarity_{column_pred}_75.parquet'
    SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

    with open(SIMILARITY_FILEPATH, 'w') as f:
        # results
        df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
        pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)  
        #   summary statistics
        df_valid_pred_all_stats = df_valid_pred_all.describe()
        f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
        df_valid_pred_all_stats.to_json(f, indent=4)
    # iterate over documents

 --------- Processing column pair: ci1_type - infrastructure_type ------------


AttributeError: 'DataFrame' object has no attribute 'chunk_id'

In [25]:
# df_valid_pred_all[:40]
df_valid_pred_all.describe()

,similarity,chunk_id_pred
count,73.000000,73.000000
mean,0.506372,1.904110
std,0.202619,1.203749
min,0.176823,0.000000
25%,0.341699,1.000000
50%,0.341699,2.000000
75%,0.661671,3.000000
max,0.790578,3.000000


### Load parquet file

In [22]:

columns_pred = ["infrastructure_type", "damage", "location"]

In [23]:
column_pred = "infrastructure_type"
SIMILARITY_FILENAME = f'llm1_similarity_{column_pred}_75.parquet'
SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    df = pd.read_parquet(SIMILARITY_FILEPATH, engine='pyarrow')
    display(df)

,impact_pred,impact_valid,similarity,citation,chunk_id_pred,is_similar
0,Buildings,roads,0.524444,EFE 2024,0,False
1,Roads,roads,1.000000,EFE 2024,0,True
2,Roads,roads,1.000000,EFE 2024,0,True
3,Energy,road,0.221013,EFE 2024,0,False
4,roads,roads,1.000000,EFE 2024,1,True
5,hospital,road,0.352201,EFE 2024,1,False
6,roads,roads,1.000000,EFE 2024,2,True
7,cars,road,0.523979,EFE 2024,2,False
8,walls,roads,0.379906,EFE 2024,2,False
9,homes,roads,0.377863,EFE 2024,2,False


## Archive - OSM-FM

In [ ]:
# import torch

# #  Fill run metrics to dictionary
# def handle_metrics(metrics, model_name, length):
#     print(f'-> Number of tags: {length}.')
#     metrics.append({
#         'modelname': model_name,
#         # 'runtime': round(end_time - start_time, 2),
#         'tagcount': length
#     })
#     return metrics


# import config
# from transformers import BertTokenizer, DistilBertTokenizer
# import random

# def serialize_data(data, tok, add_zero=False):

#     entities_tags = []

#     for d in data:

#         tags = d['tags']
#         rq_tags = {k: tags[k] for k in osm_strings.use_tags if k in tags}
#         s_tags = json_string(rq_tags)
#         entities_tags.append(s_tags)

#     entities_tags = np.array(tok.tokenize_batch(entities_tags, add_zero=add_zero))
#     return entities_tags


# class Tokenizer:

#     def __init__(self, lm):

#         if lm == 'bert':
#             self.tokenizer = BertTokenizer.from_pretrained(config.lm_names[lm])
#         elif lm == 'distilbert':
#             self.tokenizer = DistilBertTokenizer.from_pretrained(config.lm_names[lm])
#         else:
#             self.tokenizer = BertTokenizer.from_pretrained(config.lm_names[config.default_lm])

#     def tokenize_batch(self, data, add_zero=False):

#         if not data:
#             data.append(config.no_context)

#         elif add_zero:
#             data.append(config.no_context)

#         tok_data = []
#         for d in data:

#             tok_d = self.tokenizer.tokenize('[CLS] ' + d + ' [SEP]')
#             tok_data.append(tok_d)

#         tag_len = max([len(d) for d in tok_data])

#         ids_data = []
#         for tok_d in tok_data:

#             if len(tok_d) < tag_len:
#                 tok_d += ['[PAD]'] * (tag_len - len(tok_d))
#             else:
#                 tok_d = tok_d[:tag_len]

#             ids_data.append(self.tokenizer.convert_tokens_to_ids(tok_d))

#         return ids_data


# def topic_search(cd, model, topic):
#     model.eval()

#     tok = Tokenizer(config.lm)
#     topic_t = torch.tensor(
#         tok.tokenizer.convert_tokens_to_ids(tok.tokenizer.tokenize('[CLS] ' + topic + ' [SEP]'))).unsqueeze(0)
#     topic_m = torch.tensor(np.where(topic_t != 0, 1, 0))
#     emb = model.projection_p(torch.mean(model.lm(topic_t, attention_mask=topic_m)[0][:, :, :].squeeze(), 0))

#     keys = list(cd.entities.keys())
#     random.shuffle(keys)

#     for e2 in keys:
#         entity = cd.entities[e2]
#         poi = torch.tensor(serialize_data([entity], tok))
#         poi_m = torch.tensor(np.where(poi != 0, 1, 0))
#         emb2 = model.projection_p(torch.mean(model.lm(poi, attention_mask=poi_m)[0][:, :, :].squeeze(), 0))

#         print(str(topic), e2, "Similarity:", model.cos(emb, emb2).item())


In [ ]:

##################################


#  Iterate over the topics and calculate the STS for each model
for ci_impact in ci_impacts:

    metrics = []
    print(f'Calculating STS for topic/tag {ci_impact}:')
    print('--------------------------------------')

    #  Create output folder for each topic
    out_name = ci_impact        .replace('=', '_')
    out_folder = Path(s.PATH_EVAL_DATA) / 'output' / out_name
    out_folder.mkdir(exist_ok=True, parents=True)

    #  Custom models (TinyBert & Bert)
#    for lm in lm_dict.keys():
    for lm in ["tiny"]:
        print(f'Custom {lm_dict[lm].title()} model...')
        model_path = city + f'/Model/model_bert_{lm}.pkl'
        start_time = time.time()
        with open(model_path, 'rb') as f:
            if torch.cuda.is_available():
                model = pickle.load(f)
            else:
                model = CPU_Unpickler(f).load()
    output = topic_search(model, lm, topic, tags, similarity_threshold)
    end_time = time.time()
    model_name = f'custom_{lm_dict[lm].lower()}'
    write_to_file(output, out_folder, model_name)
    metrics = handle_metrics(metrics, model_name, len(output), end_time, start_time)

    #  Pre-trained models
    if use_pretrained:
        #  Pre-trained TinyBert
        lm = lm_dict['tiny']
        print(f'Pre-trained {lm.title()} model...')
        start_time = time.time()
        model = AutoModel.from_pretrained(config.lm_names[lm])
        output = topic_search_lm(model, lm, topic, tags, similarity_threshold)
        end_time = time.time()
        model_name = f'pretrained_{lm.lower()}'
        write_to_file(output, out_folder, model_name)
        metrics = handle_metrics(metrics, model_name, len(output), end_time, start_time)

        #  Pre-trained Bert
        lm = lm_dict['large']
        print(f'Pre-trained {lm.title()} model...')
        start_time = time.time()
        model = BertModel.from_pretrained(config.lm_names[lm])
        output = topic_search_lm(model, lm, topic, tags, similarity_threshold)
        end_time = time.time()
        model_name = f'pretrained_{lm.lower()}'
        write_to_file(output, out_folder, model_name)
        metrics = handle_metrics(metrics, model_name, len(output), end_time, start_time)

    print('\n')
    df = pd.DataFrame(metrics)
    df.to_csv(out_folder / 'metrics.csv', index=False)
    
print('Done!')

In [ ]:
import config
from transformers.models.bert.modeling_bert import BertModel


# #  Load the tags with the embeddings
# tags = pd.read_csv(PATH_EVAL_DATA / 'tags.csv', usecols=['key', 'value'])
# tags = [f"{key}={value}" for key, value in zip(tags['key'], tags['value'])]

# #  Iterate over the topics and calculate the STS for each model
# for topic in topics:
#     metrics = []
#     print(f'Calculating STS for topic/tag {topic}:')
#     print('--------------------------------------')

#     #  Create output folder for each topic
#     out_name = topic.replace('=', '_')
#     out_folder = Path(city) / 'output' / out_name
#     out_folder.mkdir(exist_ok=True, parents=True)

#     #  Custom models (TinyBert & Bert)
# #    for lm in lm_dict.keys():
#     for lm in ["tiny"]:
#         print(f'Custom {lm_dict[lm].title()} model...')
#         model_path = city + f'/Model/model_bert_{lm}.pkl'
#         start_time = time.time()
#         with open(model_path, 'rb') as f:
#             if torch.cuda.is_available():
#                 model = pickle.load(f)
#             else:
#                 model = CPU_Unpickler(f).load()
#         output = topic_search(model, lm, topic, tags, similarity_threshold)
#         end_time = time.time()
#         model_name = f'custom_{lm_dict[lm].lower()}'
#         write_to_file(output, out_folder, model_name)
#         metrics = handle_metrics(metrics, model_name, len(output), end_time, start_time)

#     #  Pre-trained models
#     if use_pretrained:
#         #  Pre-trained TinyBert
#         lm = lm_dict['tiny']
#         print(f'Pre-trained {lm.title()} model...')
#         start_time = time.time()
#         model = AutoModel.from_pretrained(config.lm_names[lm])
#         output = topic_search_lm(model, lm, topic, tags, similarity_threshold)
#         end_time = time.time()
#         model_name = f'pretrained_{lm.lower()}'
#         write_to_file(output, out_folder, model_name)
#         metrics = handle_metrics(metrics, model_name, len(output), end_time, start_time)

#         #  Pre-trained Bert
#         lm = lm_dict['large']
#         print(f'Pre-trained {lm.title()} model...')
#         start_time = time.time()
#         model = BertModel.from_pretrained(config.lm_names[lm])
#         output = topic_search_lm(model, lm, topic, tags, similarity_threshold)
#         end_time = time.time()
#         model_name = f'pretrained_{lm.lower()}'
#         write_to_file(output, out_folder, model_name)
#         metrics = handle_metrics(metrics, model_name, len(output), end_time, start_time)

#     print('\n')
#     df = pd.DataFrame(metrics)
#     df.to_csv(out_folder / 'metrics.csv', index=False)
    
# print('Done!')
## Merge all topic outputs and combine with reference file to compare performance
for topic in topics:
    print(f'Handling outputs for topic/tag {topic}...')
    topic_name = topic.replace('=', '_')

    in_dir = Path(city) / 'output' / topic_name
    n_rows = 25 #  How many tags of the models should be considered

    #  Load the output files
    custom_tinybert = pd.read_csv(in_dir / 'custom_tinybert_output.csv', usecols=['tag', 'rank']).set_index('tag').head(n_rows)
    custom_bert = pd.read_csv(in_dir / 'custom_bert_output.csv', usecols=['tag', 'rank']).set_index('tag').head(n_rows)
    pretrained_tinybert = pd.read_csv(in_dir / 'pretrained_tinybert_output.csv', usecols=['tag', 'rank']).set_index('tag').head(n_rows)
    pretrained_bert = pd.read_csv(in_dir / 'pretrained_bert_output.csv', usecols=['tag', 'rank']).set_index('tag').head(n_rows)

    #  Load references
    reference = pd.read_csv(PATH_EVAL_DATA / f'{topic_name}.csv', usecols=['tag']).set_index('tag')

    #  Align output files with references (keep only references)
    reference, custom_tinybert = reference.align(custom_tinybert, join='left', axis=0)
    reference, custom_bert = reference.align(custom_bert, join='left', axis=0)
    reference, pretrained_tinybert = reference.align(pretrained_tinybert, join='left', axis=0)
    reference, pretrained_bert = reference.align(pretrained_bert, join='left', axis=0)

    #  Merge all aligned dataframes and rename cols to models
    merged_df = pd.concat([reference, custom_tinybert, custom_bert, pretrained_tinybert, pretrained_bert], axis=1)
    merged_df.columns = ['custom_tinybert', 'custom_bert', 'pretrained_tinybert', 'pretrained_bert']

    #  Add empty row for separation
    merged_df.loc[''] = None

    #  Calculate the weighted harmonic sum with weights based on reference tag importance/rank
    weights = 1 / (pd.Series(range(1, len(merged_df) + 1), index=merged_df.index))
    weighted_harmonic_sum = merged_df.apply(lambda col: round((weights / col).sum(), 3), axis=0)
    merged_df.loc['Weighted Harmonic Sum'] = weighted_harmonic_sum

    #  Add true positives count to results
    counts = merged_df[:-1].notna().sum()
    new_row = {col: counts[col] for col in ['custom_tinybert', 'custom_bert', 'pretrained_tinybert', 'pretrained_bert']}
    merged_df.loc['True Positives'] = new_row

    #  Add run metrics to results
    metrics = pd.read_csv(in_dir / 'metrics.csv').set_index('modelname')
    merged_df.loc['Runtime [s]'] = metrics['runtime']
    merged_df.loc['Number of Tags'] = metrics['tagcount']

    #  File type casts
    merged_df
    #  Save to file
    out_file = in_dir / 'results.csv'
    merged_df.to_csv(out_file)
    print(f'-> Saved merged output to {out_file}.')

print('\nDone!')


FileNotFoundError: [Errno 2] No such file or directory: '../data/evaluation/manual_extracted/table_ci_impacts_sm.csv'

In [ ]:

#  Define folder for handling and writing outputs
def write_to_file(data, out_folder, filename):
    """Convert output to DataFrame and write to file"""
    df = pd.DataFrame(list(data), columns=['tag', 'sts_score'])
    #  Sort the DataFrame by similarity (explicitly)
    df = df.sort_values(by='sts_score', ascending=False)
    #  Assign integers to ranking
    df['rank'] = df['sts_score'].rank(method='first', ascending=False).astype(int)
    #  Only keep the first 20 resulting tags
    df = df.head(50)
    #  Save to file
    df.to_csv(out_folder / f'{filename}_output.csv', index=False)

#  Fill run metrics to dictionary
def handle_metrics(metrics, model_name, length, end_time, start_time):
    print(f'-> Took {end_time - start_time:.2f} seconds. Number of tags: {length}.')
    metrics.append({
        'modelname': model_name,
        'runtime': round(end_time - start_time, 2),
        'tagcount': length
    })
    return metrics

class CPU_Unpickler(pickle.Unpickler):
    """Fix for having issues with loading models on CPU"""
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        else: return super().find_class(module, name)
